# Baseline clustering — nghiên cứu

Notebook độc lập để EDA, thử feature, lọc feature, scaling/PCA, tuning và diễn giải
cụm trước khi chuyển thành pipeline trong `src`. Chạy **Restart Kernel → Run All**;
cần Internet để cài thư viện còn thiếu và tải hai CSV Google Drive.
Trước khi chạy, cấp URL qua biến môi trường `FOMO_MARKET_URL`,
`FOMO_TRADES_URL` của kernel hoặc file `configs/data.local.json` tại thư mục
hiện hành/thư mục cha. Biến môi trường được ưu tiên; không tự đọc `.env`.
Không cần `src` hay `params.yaml` để chạy notebook.

Mọi dữ liệu, model và báo cáo được giữ trong bộ nhớ. Xem `windows`,
`feature_report`, `pca_variance`, `tuning_results`, `trained_models`,
`model_evaluation`, `cluster_sizes` và `cluster_profiles` để phân tích tiếp.
Các công thức và tham số mô hình tương ứng với source hiện tại của repo.

Khi lưu sau khi chạy, output có thể nằm trong file notebook. Dùng **Clear All Outputs**
rồi lưu trước khi commit hoặc chia sẻ để tránh đưa dữ liệu mẫu vào Git.


In [ ]:
# Cài dependency còn thiếu bằng chính Python của notebook kernel.
import importlib.util
import subprocess
import sys

DEPENDENCIES = {
    'numpy': 'numpy>=2.2,<2.3',
    'pandas': 'pandas>=2.2,<3',
    'scipy': 'scipy>=1.16,<1.17',
    'sklearn': 'scikit-learn>=1.7,<1.8',
    'matplotlib': 'matplotlib>=3.9',
    'IPython': 'ipython>=8',
}
missing_packages = [spec for module, spec in DEPENDENCIES.items()
                    if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])

from io import BytesIO
import logging
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
plt.style.use('seaborn-v0_8-whitegrid')

# Cấu hình nghiên cứu; có thể chỉnh các ngưỡng và K rồi chạy lại các cell sau.
config = {'as_of_date': '2026-10-29',
 'seed': 42,
 'data': {'market_url_env': 'FOMO_MARKET_URL',
          'trades_url_env': 'FOMO_TRADES_URL'},
 'features': {'feature_set': 'fomo_32_v1',
              'window_days': 180,
              'step_days': 30,
              'min_buys': 5,
              'min_buy_days': 3},
 'preprocessing': {'min_coverage': 0.8,
                   'min_std': 0.02,
                   'min_iqr': 0.02,
                   'max_spearman': 0.7,
                   'lower_priority_features': ['order_intensity_per_active_week',
                                               'first_buy_in_rally_bias',
                                               'dip_buy_lift'],
                   'pca_variance_target': 0.9},
 'model': {'algorithms': ['kmeans', 'minibatch_kmeans', 'gmm'],
           'k_candidates': [2, 3, 4, 5, 6],
           'n_init': 5,
           'max_iter': 300}}
SEED = config['seed']
logger = logging.getLogger('baseline_research')

def require(condition, message):
    if not condition:
        raise ValueError(message)

def checked_step(name, operation, validate):
    try:
        result = operation()
        validate(result)
    except Exception:
        logger.exception('step=%s status=FAIL', name)
        raise
    logger.info('step=%s status=PASS', name)
    return result

def show_fig(_=None):
    buffer = BytesIO()
    plt.gcf().savefig(buffer, format='png', dpi=130, bbox_inches='tight')
    display(Image(data=buffer.getvalue()))
    plt.close()


## 1. Tải và kiểm tra dữ liệu

Đọc hai CSV Google Drive vào bộ nhớ, kiểm tra schema và lịch thị trường trước EDA.


In [ ]:
import json
import os
from pathlib import Path

def data_source_url(data, name):
    """Resolve a private URL without storing it in versioned parameters."""
    env_name = data[f"{name}_url_env"]
    value = os.environ.get(env_name)
    if not value:
        for directory in (Path.cwd(), Path.cwd().parent):
            local_path = directory / "configs" / "data.local.json"
            if local_path.is_file():
                with local_path.open(encoding="utf-8") as file:
                    value = json.load(file).get(env_name)
                break
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"Set {env_name} or configure configs/data.local.json")
    return value.strip()


from urllib.parse import parse_qs, urlparse
from urllib.request import urlopen

def google_drive_download_url(shared_url):
    parts = urlparse(shared_url)
    file_id = parts.path.split("/d/")[1].split("/")[0] if "/d/" in parts.path else ""
    if not file_id:
        file_id = parse_qs(parts.query).get("id", [""])[0]
    if not file_id:
        raise ValueError("Invalid Google Drive URL")
    return (
        "https://drive.usercontent.google.com/download"
        f"?id={file_id}&export=download&confirm=t"
    )

def read_drive_csv(shared_url, **kwargs):
    with urlopen(google_drive_download_url(shared_url)) as response:
        return pd.read_csv(BytesIO(response.read()), **kwargs)


def load_research_data(config):
    market_input = read_drive_csv(data_source_url(config['data'], 'market'))
    trades_input = read_drive_csv(data_source_url(config['data'], 'trades'),
                                 dtype={'order_id': str, 'investor_id': str})
    return market_input, trades_input


def check_raw_data(result):
    market_input, trades_input = result
    schemas = [
        (market_input, {'timestamp', 'open', 'high', 'low', 'close', 'volume'}),
        (trades_input, {'order_id', 'timestamp', 'investor_id', 'side', 'price',
                        'quantity', 'amount', 'fee', 'cash_after'}),
    ]
    for frame, required in schemas:
        require(not frame.empty, 'Raw data is empty')
        require(required <= set(frame.columns), f'Missing columns: {required - set(frame.columns)}')
        require(pd.to_datetime(frame.timestamp, errors='raise').notna().all(), 'Missing timestamps')
    dates = pd.to_datetime(market_input.timestamp).dt.normalize().sort_values()
    require(not dates.duplicated().any(), 'Duplicate market dates')
    require(len(dates) == len(pd.date_range(dates.min(), dates.max())), 'Missing market dates')
    require(trades_input.investor_id.notna().all(), 'Missing investor ID')
    require(trades_input.side.str.strip().str.upper().isin(['BUY', 'SELL']).all(), 'Invalid trade side')


In [ ]:
# Giữ bản input nguyên gốc cho FE; EDA dùng các bản copy đã bổ sung cột.
market_input, trades_input = checked_step(
    'extract', lambda: load_research_data(config), check_raw_data,
)
market = market_input.copy()
trades = trades_input.copy()
market['timestamp'] = pd.to_datetime(market['timestamp'])
trades['timestamp'] = pd.to_datetime(trades['timestamp'])
market['date'] = market['timestamp'].dt.normalize()
market = market.set_index('date').sort_index()
trades['date'] = trades['timestamp'].dt.normalize()
trades['side'] = trades['side'].str.strip().str.upper()
trades['is_buy'] = trades['side'].eq('BUY')
trades['buy_amount'] = trades['amount'].where(trades['is_buy'], 0.0)
trades['sell_amount'] = trades['amount'].where(~trades['is_buy'], 0.0)
trades['buy_cost'] = (trades['amount'] + trades['fee']).where(trades['is_buy'], 0.0)
trades['sell_proceeds'] = (trades['amount'] - trades['fee']).where(~trades['is_buy'], 0.0)
trades = trades.sort_values(['investor_id', 'timestamp', 'order_id']).reset_index(drop=True)
trades['cash_before'] = np.where(
    trades['is_buy'],
    trades['cash_after'] + trades['amount'] + trades['fee'],
    trades['cash_after'] - trades['amount'] + trades['fee']
)
context = pd.DataFrame(index=market.index)
context['past_return'] = market['close'].pct_change(7, fill_method=None).shift(1)
context['prior_high'] = market['high'].rolling(90, min_periods=90).max().shift(1)
context['rally_known'] = context['past_return'].notna()
context['near_known'] = context['prior_high'].notna()
context['rally'] = context['past_return'].ge(0.10)
context['near_high'] = market['close'].shift(1).div(context['prior_high']).ge(0.95)
raw_market = market.copy()
raw_trades = trades.copy()
display(market.head())
display(trades.head())


## 2. EDA


In [ ]:
# 1. Chuỗi thời gian giữ đúng thứ tự ngày, phù hợp để nhìn regime và xu hướng giá.
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(raw_market.index, raw_market.close, color='#2878a0', lw=1.6)
ax.set(title='Giá đóng cửa BTC theo thời gian', xlabel='Ngày', ylabel='USD')
plt.tight_layout(); show_fig('01a_btc_price_timeline.png')


In [ ]:
# 2. Histogram cho thấy phân phối số ngày hoạt động giữa các nhà đầu tư.
investor_activity = raw_trades.groupby('investor_id').agg(n_trades=('order_id', 'size'), n_buys=('is_buy', 'sum'), active_days=('date', 'nunique'))
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(investor_activity.active_days, bins=50, color='#90be6d', edgecolor='white')
ax.axvline(investor_activity.active_days.median(), color='#333333', ls='--', label=f'Trung vị = {investor_activity.active_days.median():.0f} ngày')
ax.set(title='Phân phối số ngày hoạt động của nhà đầu tư', xlabel='Số ngày có giao dịch', ylabel='Số nhà đầu tư'); ax.legend()
plt.tight_layout(); show_fig('01b_investor_active_days.png')


In [ ]:
# 3. Log10 làm co đuôi phải của amount để toàn bộ dải quy mô xuất hiện trên cùng hình.
buy_trades = raw_trades[raw_trades.is_buy].copy()
log_buy_amount = np.log10(buy_trades.amount)
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(log_buy_amount, bins=60, color='#f9c74f', edgecolor='white')
ax.set(title='Phân phối quy mô lệnh mua trên thang log10', xlabel='log10(amount USD)', ylabel='Số lệnh mua')
plt.tight_layout(); show_fig('01c_buy_amount_log.png')


In [ ]:
# 4. Bar chart so sánh trực tiếp số cơ hội rally và near-high giữa các năm.
context_year = context.groupby(context.index.year).agg(rally_days=('rally', 'sum'), near_high_days=('near_high', 'sum'))
fig, ax = plt.subplots(figsize=(10, 4))
context_year.plot.bar(ax=ax, color=['#e76f51', '#457b9d'])
ax.set(title='Số ngày xuất hiện từng trạng thái thị trường theo năm', xlabel='Năm', ylabel='Số ngày'); ax.legend(['Rally', 'Gần đỉnh 90 ngày'])
plt.tight_layout(); show_fig('01d_market_opportunities_by_year.png')


In [ ]:
# 5. Histogram lợi suất tuần và vạch giả thuyết rally.
weekly_return = market.close.pct_change(7, fill_method=None).dropna()
rally_share = weekly_return.ge(.10).mean()
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(weekly_return, bins=60, color='#2878a0', edgecolor='white')
ax.axvline(.10, color='#e76f51', ls='--', label=f'Ngưỡng rally 10% ({rally_share:.1%} ngày)')
ax.set(title='Phân phối lợi suất BTC trong 7 ngày', xlabel='Lợi suất 7 ngày', ylabel='Số ngày'); ax.legend()
plt.tight_layout(); show_fig('02a_weekly_return.png')


In [ ]:
# 6. Khoảng cách giữa hai ngày mua liên tiếp theo từng nhà đầu tư.
buy_dates = buy_trades.sort_values(['investor_id', 'date'])
buy_gaps = buy_dates.groupby('investor_id')['date'].diff().dt.days.dropna()
buy_gaps_plot = buy_gaps.clip(upper=180)
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(buy_gaps_plot, bins=45, color='#90be6d', edgecolor='white')
ax.axvline(buy_gaps.median(), color='#333333', ls='--', label=f'Trung vị = {buy_gaps.median():.0f} ngày')
ax.set(title='Khoảng cách giữa hai ngày mua liên tiếp', xlabel='Số ngày, giá trị trên 180 được gộp tại 180', ylabel='Số khoảng mua'); ax.legend()
plt.tight_layout(); show_fig('02b_inter_buy_gap.png')


In [ ]:
# 7. Log10 số dư tiền mặt trước lệnh và kiểm tra tính liên tục dòng tiền.
cash_error = raw_trades['cash_before'] - raw_trades.groupby('investor_id')['cash_after'].shift(1)
log_cash = np.log10(raw_trades.cash_before.clip(lower=1e-8))
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(log_cash, bins=50, color='#f9c74f', edgecolor='white')
ax.set(title='Phân phối tiền mặt trước giao dịch trên thang log10', xlabel='log10(cash_before USD)', ylabel='Số giao dịch')
plt.tight_layout(); show_fig('02c_cash_before_log.png')


In [ ]:
# 8. Hướng giao dịch thay đổi theo mức lợi suất thị trường.
trade_context = raw_trades.join(context[['past_return','rally','near_high']], on='date')
return_bins = [-np.inf, -.10, -.03, .03, .10, np.inf]
return_labels = ['< -10%', '-10% đến -3%', '-3% đến 3%', '3% đến 10%', '>= 10%']
trade_context['return_group'] = pd.cut(trade_context.past_return, bins=return_bins, labels=return_labels, right=False)
return_behavior = trade_context.dropna(subset=['return_group']).groupby('return_group', observed=True).agg(
    buy_share=('is_buy','mean'), n_trades=('order_id','size')).reindex(return_labels)
fig, ax = plt.subplots(figsize=(10,4.5))
bars=ax.bar(return_behavior.index.astype(str), return_behavior.buy_share, color=['#457b9d','#74a9cf','#bdbdbd','#f4a261','#e76f51'])
for b,v,n_obs in zip(bars,return_behavior.buy_share,return_behavior.n_trades):
    ax.text(b.get_x()+b.get_width()/2,v+.012,f'{v:.1%}\nn={n_obs:,}',ha='center',va='bottom',fontsize=9)
ax.axhline(trade_context.is_buy.mean(),color='#333',ls='--',label='Tỷ lệ mua chung trên ngày giao dịch')
ax.set(title='Tỷ lệ lệnh mua theo vùng lợi suất BTC 7 ngày',xlabel='Lợi suất đã biết trước ngày giao dịch',ylabel='Tỷ lệ BUY',ylim=(0,1)); ax.legend()
plt.tight_layout(); show_fig('02d_buy_share_by_return.png')


In [ ]:
# 9. So sánh hướng giao dịch ở gần đỉnh và xa đỉnh.
near_behavior = trade_context.loc[trade_context.date.map(context.near_known).fillna(False)].groupby('near_high').agg(buy_share=('is_buy','mean'),n_trades=('order_id','size'))
fig, ax = plt.subplots(figsize=(8,4.5))
labels=['Xa đỉnh 90 ngày','Trong 5% dưới đỉnh 90 ngày']; vals=[near_behavior.loc[False,'buy_share'],near_behavior.loc[True,'buy_share']]
bars=ax.bar(labels,vals,color=['#457b9d','#e76f51'])
for b,v,key in zip(bars,vals,[False,True]): ax.text(b.get_x()+b.get_width()/2,v+.012,f'{v:.1%}\nn={near_behavior.loc[key,"n_trades"]:,}',ha='center')
ax.set(title='Tỷ lệ BUY theo vị trí so với đỉnh 90 ngày',ylabel='Tỷ lệ BUY',ylim=(0,1))
plt.tight_layout(); show_fig('02e_buy_share_near_high.png')


In [ ]:
# 10. Độ dài chuỗi mua ở các ngày liên tiếp.
run_lengths=[]
for _,g in buy_dates.groupby('investor_id'):
    gaps=g.date.diff().dt.days
    run_id=gaps.ne(1).cumsum()
    run_lengths.extend(g.groupby(run_id).size().tolist())
run_lengths=pd.Series(run_lengths)
run_dist=run_lengths.clip(upper=10).value_counts(normalize=True).sort_index()
fig, ax = plt.subplots(figsize=(9,4.5))
bars=ax.bar(run_dist.index.astype(str),run_dist.values,color='#f4a261')
ax.set(title='Độ dài chuỗi ngày mua liên tiếp',xlabel='Số ngày liên tiếp; 10 gồm cả chuỗi dài hơn',ylabel='Tỷ lệ chuỗi')
for b,v in zip(bars,run_dist.values): ax.text(b.get_x()+b.get_width()/2,v+.006,f'{v:.1%}',ha='center',fontsize=8)
plt.tight_layout(); show_fig('02f_consecutive_buy_runs.png')


In [ ]:
buy_with_gap = buy_dates.copy()
buy_with_gap['gap_days'] = buy_with_gap.groupby('investor_id')['date'].diff().dt.days
buy_with_gap = buy_with_gap.join(context['rally'], on='date').dropna(subset=['gap_days'])

median_gap = buy_with_gap.groupby('rally')['gap_days'].median().reindex([False, True])

plt.figure(figsize=(7, 4))
bars = plt.bar(['Không rally', 'Rally'], median_gap, color=['#457b9d', '#e76f51'])

for bar, value in zip(bars, median_gap):
    plt.text(bar.get_x() + bar.get_width()/2, value, f'{value:.0f} ngày',
             ha='center', va='bottom')

plt.title('Khoảng nghỉ trung vị trước lệnh mua')
plt.ylabel('Số ngày')
plt.tight_layout()
plt.show()

In [ ]:
# 12. Tỷ lệ giải ngân trên tiền có sẵn.
buy_capital=buy_trades.copy(); buy_capital['deployment_ratio']=(buy_capital.buy_cost/buy_capital.cash_before.where(buy_capital.cash_before>0)).clip(0,1)
fig, ax = plt.subplots(figsize=(9,4.5))
ax.hist(buy_capital.deployment_ratio.dropna(),bins=np.linspace(0,1,41),color='#e76f51',edgecolor='white')
ax.axvline(.5,color='#333',ls='--',label=f'>=50%: {buy_capital.deployment_ratio.ge(.5).mean():.1%}')
ax.axvline(.9,color='#7b2cbf',ls='--',label=f'>=90%: {buy_capital.deployment_ratio.ge(.9).mean():.1%}')
ax.set(title='Tỷ lệ giá trị mua trên tiền mặt trước lệnh',xlabel='(amount + fee) / cash_before',ylabel='Số lệnh BUY'); ax.legend()
plt.tight_layout(); show_fig('02h_cash_deployment_ratio.png')


In [ ]:
# 13. Cân bằng mua và bán khác nhau giữa các nhà đầu tư.
side_profile=raw_trades.groupby('investor_id').agg(buy_share=('is_buy','mean'),n_trades=('order_id','size'))
fig, ax = plt.subplots(figsize=(9,4.5))
ax.hist(side_profile.buy_share,bins=np.linspace(0,1,31),color='#577590',edgecolor='white')
ax.axvline(.5,color='#333',ls='--',label='Mua và bán cân bằng')
ax.set(title='Tỷ lệ BUY trong toàn bộ giao dịch của từng nhà đầu tư',xlabel='Số lệnh BUY / tổng số lệnh',ylabel='Số nhà đầu tư'); ax.legend()
plt.tight_layout(); show_fig('02i_investor_buy_sell_balance.png')


In [ ]:
incidence = []

for _, trades_by_investor in raw_trades.groupby('investor_id'):
    days = context.loc[trades_by_investor.date.min():]
    days = days[days.rally_known & days.near_known]
    bought = pd.Series(days.index.isin(trades_by_investor.loc[trades_by_investor.is_buy, 'date']), index=days.index)

    incidence.append({
        state: bought[days[state]].mean() - bought[~days[state]].mean()
        for state in ['rally', 'near_high']
    })

incidence = pd.DataFrame(incidence)
median_incidence = incidence.median()

plt.figure(figsize=(7, 4))
bars = plt.bar(
    ['Rally − ngoài rally', 'Gần đỉnh − xa đỉnh'],
    median_incidence,
    color=['#457b9d', '#e76f51']
)
plt.axhline(0, color='gray', linestyle='--')

for bar, value in zip(bars, median_incidence):
    plt.text(bar.get_x() + bar.get_width()/2, value, f'{value:.1%}', ha='center', va='bottom')

plt.title('Chênh lệch tỷ lệ ngày mua trung vị')
plt.ylabel('Chênh lệch tỷ lệ ngày mua')
plt.tight_layout()
plt.show()



### Kết luận EDA

- Giá BTC thay đổi qua nhiều mặt bằng theo thời gian, vì vậy nên dùng trạng thái giá tương đối thay cho giá USD tuyệt đối.
- Phần lớn nhà đầu tư chỉ hoạt động trong ít ngày, còn một nhóm nhỏ giao dịch kéo dài; số lệnh thô dễ phản ánh mức hoạt động hơn là hành vi.
- Quy mô lệnh mua chênh lệch rất lớn giữa các tài khoản, nên cần chuẩn hóa theo vốn hoặc lịch sử riêng của từng người.
- Số ngày rally và gần đỉnh thay đổi theo năm, nên hành vi mua phải được tính trên số ngày có cơ hội tương ứng.
- Rally từ 10% trong 7 ngày nằm ở phần đuôi nhưng vẫn có đủ quan sát để dùng làm trạng thái so sánh.
- Khoảng cách giữa các lần mua có cả chuỗi mua sát nhau và thời gian nghỉ dài, cho thấy nhịp mua không đều.
- Tiền mặt giữa các tài khoản khác nhau lớn nhưng dòng tiền liên tục, vì vậy có thể dùng tỷ lệ giải ngân trên tiền sẵn có.
- Tỷ lệ BUY thay đổi theo lợi suất 7 ngày, cho thấy hướng giao dịch có liên hệ với đà thị trường.
- Tỷ lệ BUY gần đỉnh khác khi xa đỉnh, nên vị trí so với đỉnh 90 ngày có ích cho việc mô tả hành vi.
- Một số nhà đầu tư mua nhiều ngày liên tiếp, cho thấy chuỗi mua chứa thêm thông tin ngoài tổng số lệnh.
- Khoảng nghỉ trước lệnh mua khác nhau giữa ngày rally và ngày bình thường, nhưng chưa đủ để kết luận tâm lý FOMO.
- Phần vốn giải ngân trên mỗi lệnh phân tán mạnh, trong khi hành vi dùng gần hết tiền xuất hiện khá hiếm.
- Tỷ lệ mua và bán khác nhau giữa nhà đầu tư, nhưng dữ liệu chưa đủ để khẳng định động cơ cắt lỗ hay chốt lời.
- Khi cho mỗi nhà đầu tư trọng số ngang nhau, xu hướng mua trong rally và gần đỉnh vẫn xuất hiện nhưng chỉ là quan hệ quan sát.


## 3. Window

So sánh độ dài và bước trượt để chọn cửa sổ đủ dữ liệu nhưng không tạo quá nhiều lần cập nhật lặp.


In [ ]:
# EDA des calendriers: mêmes dates de fin pour comparer les durées.
import numpy as np
import pandas as pd

def evaluate_calendars(trades, context, lengths=(60,90,120,150,180,240,365)):
    dates=context.index
    ids=pd.Index(sorted(trades.investor_id.unique()))
    purchases=np.zeros((len(ids),len(dates)),dtype=np.int32)
    buys=trades.loc[trades.is_buy]
    np.add.at(purchases,(ids.get_indexer(buys.investor_id),dates.get_indexer(buys.date)),1)
    cumulative=np.pad(purchases.cumsum(axis=1),((0,0),(1,0)))
    first=dates.get_indexer(trades.groupby('investor_id').date.min().reindex(ids))
    # Toutes les fenêtres comparées débutent après l'historique de 90 jours.
    valid_start=np.flatnonzero(context.rally_known & context.near_known)[0]
    common_ends=np.arange(valid_start+max(lengths)-1,len(dates))
    known=context.rally_known & context.near_known
    states=np.column_stack([known & context.rally,known & ~context.rally,
                            known & context.near_high,known & ~context.near_high]).astype(int)
    cs=np.vstack([np.zeros(4,int),states.cumsum(axis=0)])
    rows=[]
    for w in lengths:
        end=common_ends; start=end-w+1
        history=first[:,None]<=start
        counts=cumulative[:,end+1]-cumulative[:,start]
        market_ok=((cs[end+1]-cs[start])>=5).all(axis=1)
        rows.append(dict(days=w,market_coverage=market_ok.mean(),
            support=(counts[history]>=5).mean(),n_eligible=int(history.sum())))
    designs=[]
    # Rolling, tumbling, expanding, et fenêtres limitées à chaque année.
    for step in [7,14,30,60,90,180]:
        ends=np.arange(valid_start+179,len(dates),step)
        designs.append((f'180/{step}',ends-179,ends))
    ends=np.arange(valid_start+179,len(dates),30)
    designs.append(('Mở rộng/30',np.full(len(ends),valid_start),ends))
    starts=[]; ends=[]
    for yr in np.unique(dates.year):
        ix=np.flatnonzero((dates.year==yr)&(np.arange(len(dates))>=valid_start))
        if len(ix)>=180:
            ee=np.arange(ix[0]+179,ix[-1]+1,30)
            starts.extend(ee-179); ends.extend(ee)
    designs.append(('180/30 reset năm',np.array(starts),np.array(ends)))
    out=[]
    for name,start,end in designs:
        counts=cumulative[:,end+1]-cumulative[:,start]
        history=first[:,None]<=start
        support=history & (counts>=5)
        cover=np.zeros(len(dates),bool)
        for a,b in zip(start,end): cover[a:b+1]=True
        adjacent=history[:,1:] & history[:,:-1]
        new_counts=cumulative[:,end[1:]+1]-cumulative[:,end[:-1]+1]
        no_new=(new_counts[adjacent]==0).mean()
        out.append(dict(design=name,n_windows=int(history.sum()),support=support.sum()/history.sum(),
            no_new_buy=no_new,calendar_coverage=cover[valid_start:].mean(),
            cross_year=np.mean(dates[start].year!=dates[end].year),
            median_update=float(np.median(np.diff(end)))))
    return pd.DataFrame(rows),pd.DataFrame(out)

window_audit, calendar_audit=evaluate_calendars(raw_trades,context)
fig,ax=plt.subplots(figsize=(11,4))
x=np.arange(len(window_audit))
ax.bar(x-.18,window_audit.market_coverage,.36,label='Đủ 4 trạng thái thị trường')
ax.bar(x+.18,window_audit.support,.36,label='Có ít nhất 5 lệnh mua')
ax.axhline(.9,color='gray',ls='--'); ax.set_xticks(x,window_audit.days.astype(str)); ax.set(xlabel='Độ dài (ngày)',ylabel='Tỷ lệ đạt yêu cầu',ylim=(0,1.12)); ax.legend()
show_fig('03e_common_end_windows.png')
fig,ax=plt.subplots(figsize=(12,4))
x=np.arange(len(calendar_audit))
ax.bar(x-.18,calendar_audit.no_new_buy,.36,label='Không có BUY mới từ lần cập nhật trước')
ax.bar(x+.18,calendar_audit.support,.36,label='Đủ ít nhất 5 BUY trong cửa sổ')
ax.set_xticks(x,calendar_audit.design,rotation=20,ha='right'); ax.set(ylabel='Tỷ lệ',ylim=(0,1.15)); ax.legend()
show_fig('03f_overlap_vs_tumbling.png')
fig,ax=plt.subplots(figsize=(12,4))
x=np.arange(len(calendar_audit))
ax.bar(x,calendar_audit.n_windows,color='#2878a0'); ax.set_xticks(x,calendar_audit.design,rotation=20,ha='right'); ax.set(ylabel='Số cửa sổ đủ lịch sử')
for i,r in calendar_audit.iterrows(): ax.text(i,r.n_windows,f'{r.calendar_coverage:.0%} ngày\n{r.median_update:g} ngày/lượt',ha='center',va='bottom',fontsize=8)
ax.margins(y=.25); show_fig('03g_window_cost_coverage.png')


## 4. Feature engineering

Tạo 32 feature trực tiếp từ các DataFrame raw trong bộ nhớ, cùng công thức và bộ lọc với source.


In [ ]:
"""Create the 32 rolling-window features used by the baseline notebook.

Six features retain the executable Week 1 formulas. The other 26 reconstruct
the definitions in Clustering_Presentation.md because their Python source is
not present. Every market signal uses information available by t-1.
"""


# pyrefly: ignore[missing-import]
import numpy as np
import pandas as pd


CANDIDATE_32 = [
    "rally_buy_lift", "near_high_buy_lift", "ath_proximity_buy_ratio",
    "green_streak_buy_bias", "volume_surge_buy_lift", "high_volatility_buy_lift",
    "rally_size_excess", "near_high_size_lift", "rally_cash_excess",
    "max_cash_deployment_ratio", "all_in_frequency", "net_buy_cash_ratio",
    "rally_burst_excess", "consecutive_buy_run_ratio", "buy_frequency_acceleration",
    "inter_buy_gap_cv", "weekend_buy_share", "order_intensity_per_active_week",
    "rally_reactivation_excess", "dormant_breakout_buy_rate", "fast_reentry_rate",
    "inactivity_to_rally_ratio", "first_buy_in_rally_bias", "dip_buy_lift",
    "fomo_vs_dip_ratio", "pullback_entry_bias", "buy_sell_momentum_imbalance",
    "rally_sell_suppression", "sell_to_buy_day_ratio",
    "portfolio_turnover_velocity", "panic_sell_post_top_rate",
    "loss_cut_vs_profit_exit_ratio",
]

REQUIRED_MARKET_COLUMNS = {"timestamp", "open", "high", "low", "close", "volume"}
REQUIRED_TRADE_COLUMNS = {
    "order_id", "timestamp", "investor_id", "side", "price", "quantity",
    "amount", "fee", "cash_after",
}

RETURN_DAYS, RALLY_RETURN = 7, 0.10
HIGH_LOOKBACK, NEAR_HIGH_RATIO = 90, 0.95
WINDOW_DAYS, STEP_DAYS = 180, 30
BASELINE_DAYS, MIN_HISTORY_DAYS, MIN_BASELINE_BUYS = 90, 30, 3
MIN_CONTEXT_DAYS, MIN_CONDITION_BUY_DAYS = 5, 2
BURST_DAYS, CASH_DAYS = 7, 3
GREEN_STREAK_DAYS, VOLUME_LOOKBACK, VOLATILITY_LOOKBACK = 3, 30, 30
DIP_RETURN, FAST_REENTRY_DAYS, PANIC_SELL_DAYS, ALL_IN_RATIO = -0.05, 7, 7, 0.80


def _check_columns(frame, required, label):
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{label} missing columns: {sorted(missing)}")


def _ratio(numerator, denominator):
    if denominator is None or not np.isfinite(denominator) or denominator == 0:
        return np.nan
    return float(numerator / denominator)


def _bounded_change(current, baseline):
    denominator = current + baseline
    return (current - baseline) / denominator.where(denominator > 0)


def _contrast(frame, value, state, known, minimum):
    good = known.fillna(False) & frame[value].notna()
    yes, no = good & state.fillna(False), good & ~state.fillna(False)
    if int(yes.sum()) < minimum or int(no.sum()) < minimum:
        return np.nan
    return float(frame.loc[yes, value].mean() - frame.loc[no, value].mean())


def _market_context(market):
    market = market.copy()
    market["date"] = pd.to_datetime(market["timestamp"]).dt.normalize()
    market = market.set_index("date").sort_index()
    if market.index.duplicated().any():
        raise ValueError("Market data contains duplicate dates")

    context = pd.DataFrame(index=market.index)
    daily_return = market["close"].pct_change(fill_method=None)
    context["past_return"] = market["close"].pct_change(
        RETURN_DAYS, fill_method=None
    ).shift(1)
    context["prior_high"] = market["high"].rolling(
        HIGH_LOOKBACK, min_periods=HIGH_LOOKBACK
    ).max().shift(1)
    context["rally_known"] = context["past_return"].notna()
    context["near_known"] = context["prior_high"].notna()
    context["rally"] = context["past_return"].ge(RALLY_RETURN)
    context["near_high"] = market["close"].shift(1).div(context["prior_high"]).ge(
        NEAR_HIGH_RATIO
    )

    positive = daily_return.gt(0).astype(float)
    context["green_streak"] = positive.shift(1).rolling(
        GREEN_STREAK_DAYS, min_periods=GREEN_STREAK_DAYS
    ).sum().eq(GREEN_STREAK_DAYS)
    volume_median = market["volume"].shift(1).rolling(
        VOLUME_LOOKBACK, min_periods=VOLUME_LOOKBACK
    ).median()
    context["volume_surge"] = market["volume"].shift(1).gt(1.5 * volume_median)
    true_range = (market["high"] - market["low"]).div(market["open"])
    volatility_limit = true_range.shift(1).rolling(
        VOLATILITY_LOOKBACK, min_periods=VOLATILITY_LOOKBACK
    ).quantile(0.75)
    context["high_volatility"] = true_range.shift(1).gt(volatility_limit)
    context["dip"] = context["past_return"].le(DIP_RETURN)
    context["pullback"] = context["past_return"].gt(0) & daily_return.shift(1).lt(0)
    context["momentum"] = context["past_return"].abs().ge(0.05)
    context["left_top"] = context["near_high"].shift(1).eq(True) & ~context["near_high"]
    context["post_top"] = context["left_top"].shift(1).rolling(
        PANIC_SELL_DAYS, min_periods=1
    ).max().fillna(0).astype(bool)
    return market, context


def _daily_trades(trades):
    trades = trades.copy()
    trades["timestamp"] = pd.to_datetime(trades["timestamp"])
    trades["date"] = trades["timestamp"].dt.normalize()
    trades["side"] = trades["side"].str.strip().str.upper()
    if not trades["side"].isin(["BUY", "SELL"]).all():
        raise ValueError("Trade side must be BUY or SELL")
    trades = trades.sort_values(["investor_id", "timestamp", "order_id"])
    trades["is_buy"] = trades["side"].eq("BUY")
    trades["is_sell"] = trades["side"].eq("SELL")
    trades["buy_amount"] = trades["amount"].where(trades["is_buy"], 0.0)
    trades["sell_amount"] = trades["amount"].where(trades["is_sell"], 0.0)
    trades["buy_cost"] = (trades["amount"] + trades["fee"]).where(trades["is_buy"], 0.0)
    trades["sell_proceeds"] = (trades["amount"] - trades["fee"]).where(trades["is_sell"], 0.0)
    trades["cash_before"] = np.where(
        trades["is_buy"],
        trades["cash_after"] + trades["amount"] + trades["fee"],
        trades["cash_after"] - trades["amount"] + trades["fee"],
    )

    prior_value = trades["buy_amount"].groupby(trades["investor_id"]).cumsum() - trades["buy_amount"]
    buy_quantity = trades["quantity"].where(trades["is_buy"], 0.0)
    prior_quantity = buy_quantity.groupby(trades["investor_id"]).cumsum() - buy_quantity
    trades["reference_buy_price"] = prior_value / prior_quantity.replace(0, np.nan)
    trades["loss_exit"] = trades["is_sell"] & trades["price"].lt(trades["reference_buy_price"])
    trades["profit_exit"] = trades["is_sell"] & trades["price"].ge(trades["reference_buy_price"])

    return trades.groupby(["investor_id", "date"], sort=False).agg(
        n_trades=("order_id", "size"), n_buys=("is_buy", "sum"),
        n_sells=("is_sell", "sum"), buy_amount=("buy_amount", "sum"),
        sell_amount=("sell_amount", "sum"), buy_cost=("buy_cost", "sum"),
        sell_proceeds=("sell_proceeds", "sum"), first_cash=("cash_before", "first"),
        cash_after=("cash_after", "last"), loss_exits=("loss_exit", "sum"),
        profit_exits=("profit_exit", "sum"),
    )


def _prepare_daily(raw, calendar, context):
    first_date = raw.index.min()
    daily = raw.reindex(calendar[calendar >= first_date]).copy().join(context)
    zero_columns = [
        "n_trades", "n_buys", "n_sells", "buy_amount", "sell_amount",
        "buy_cost", "sell_proceeds", "loss_exits", "profit_exits",
    ]
    daily[zero_columns] = daily[zero_columns].fillna(0)
    daily["buy_day"] = daily["n_buys"].gt(0)
    daily["sell_day"] = daily["n_sells"].gt(0)
    daily["active_day"] = daily["n_trades"].gt(0)
    daily["cash_after"] = daily["cash_after"].ffill()
    daily["cash_start"] = daily["cash_after"].shift(1)
    if pd.notna(daily.loc[first_date, "first_cash"]):
        daily.loc[first_date, "cash_start"] = daily.loc[first_date, "first_cash"]

    prior_n = daily["n_buys"].shift(1).rolling(
        BASELINE_DAYS, min_periods=MIN_HISTORY_DAYS
    ).sum()
    prior_amount = daily["buy_amount"].shift(1).rolling(
        BASELINE_DAYS, min_periods=MIN_HISTORY_DAYS
    ).sum()
    prior_mean = prior_amount / prior_n.where(prior_n >= MIN_BASELINE_BUYS)
    order_mean = daily["buy_amount"] / daily["n_buys"].where(daily["n_buys"] > 0)
    daily["size_change"] = _bounded_change(order_mean, prior_mean)

    past_buys = daily["n_buys"].shift(BURST_DAYS).rolling(
        BASELINE_DAYS, min_periods=MIN_HISTORY_DAYS
    )
    baseline_rate = past_buys.mean().where(past_buys.sum() >= MIN_BASELINE_BUYS)
    recent_count = daily["n_buys"].rolling(BURST_DAYS, min_periods=BURST_DAYS).sum()
    daily["burst_change"] = _bounded_change(recent_count, BURST_DAYS * baseline_rate)

    net_spend = (daily["buy_cost"] - daily["sell_proceeds"]).rolling(
        CASH_DAYS, min_periods=CASH_DAYS
    ).sum()
    cash_at_start = daily["cash_start"].shift(CASH_DAYS - 1)
    daily["cash_deployment"] = (
        net_spend / cash_at_start.where(cash_at_start > 0)
    ).clip(0, 1)

    dates = pd.Series(daily.index, index=daily.index)
    last_buy = dates.where(daily["buy_day"]).ffill().shift(1)
    daily["buy_gap"] = (dates - last_buy).dt.days.clip(upper=BASELINE_DAYS)
    history = daily["n_buys"].shift(1).rolling(
        BASELINE_DAYS, min_periods=MIN_HISTORY_DAYS
    )
    daily["expected_gap"] = (
        1 / history.mean().where(history.sum() >= MIN_BASELINE_BUYS)
    ).clip(upper=BASELINE_DAYS)
    daily["reactivation_change"] = _bounded_change(
        daily["buy_gap"], daily["expected_gap"]
    ).where(daily["buy_day"])
    daily["dormant_buy"] = daily["buy_day"] & daily["buy_gap"].gt(daily["expected_gap"])
    last_sell = dates.where(daily["sell_day"]).ffill().shift(1)
    daily["days_since_sell"] = (dates - last_sell).dt.days
    return daily


def _run_ratio(values):
    values = np.asarray(values, dtype=bool)
    if values.sum() == 0:
        return np.nan
    previous, following = np.r_[False, values[:-1]], np.r_[values[1:], False]
    return float((values & (previous | following)).sum() / values.sum())


def _window_features(window):
    buy, sell = window["buy_day"], window["sell_day"]
    n_buy, n_sell = int(buy.sum()), int(sell.sum())
    features = {
        "rally_buy_lift": _contrast(window, "buy_day", window["rally"], window["rally_known"], MIN_CONTEXT_DAYS),
        "near_high_buy_lift": _contrast(window, "buy_day", window["near_high"], window["near_known"], MIN_CONTEXT_DAYS),
        "rally_size_excess": _contrast(window, "size_change", window["rally"], window["rally_known"] & buy, MIN_CONDITION_BUY_DAYS),
        "rally_cash_excess": _contrast(window, "cash_deployment", window["rally"], window["rally_known"] & buy, MIN_CONDITION_BUY_DAYS),
        "rally_burst_excess": 0.5 * _contrast(window, "burst_change", window["rally"], window["rally_known"], MIN_CONTEXT_DAYS),
        "rally_reactivation_excess": _contrast(window, "reactivation_change", window["rally"], window["rally_known"] & buy, MIN_CONDITION_BUY_DAYS),
        "ath_proximity_buy_ratio": _ratio((buy & window["near_high"]).sum(), n_buy),
        "green_streak_buy_bias": _contrast(window, "buy_day", window["green_streak"], window["rally_known"], MIN_CONTEXT_DAYS),
        "volume_surge_buy_lift": _contrast(window, "buy_day", window["volume_surge"], window["rally_known"], MIN_CONTEXT_DAYS),
        "high_volatility_buy_lift": _contrast(window, "buy_day", window["high_volatility"], window["rally_known"], MIN_CONTEXT_DAYS),
        "near_high_size_lift": _contrast(window, "size_change", window["near_high"], window["near_known"] & buy, MIN_CONDITION_BUY_DAYS),
        "max_cash_deployment_ratio": float(window.loc[buy, "cash_deployment"].max()) if buy.any() else np.nan,
        "all_in_frequency": _ratio((buy & window["cash_deployment"].ge(ALL_IN_RATIO)).sum(), n_buy),
        "consecutive_buy_run_ratio": _run_ratio(buy),
        "inter_buy_gap_cv": np.nan,
        "weekend_buy_share": _ratio((buy & (window.index.dayofweek >= 5)).sum(), n_buy),
        "sell_to_buy_day_ratio": _ratio(n_sell, n_buy),
        "loss_cut_vs_profit_exit_ratio": _ratio(window["loss_exits"].sum(), window["loss_exits"].sum() + window["profit_exits"].sum()),
    }

    buy_dates = window.index[buy]
    if len(buy_dates) >= 3:
        gaps = pd.Series(buy_dates).diff().dt.days.dropna()
        features["inter_buy_gap_cv"] = _ratio(gaps.std(ddof=0), gaps.mean())
    half = len(window) // 2
    early, late = window["buy_day"].iloc[:half].mean(), window["buy_day"].iloc[half:].mean()
    features["buy_frequency_acceleration"] = (late - early) / (late + early) if late + early > 0 else np.nan
    active_weeks = window.loc[window["active_day"]].index.to_period("W").nunique()
    features["order_intensity_per_active_week"] = _ratio(window["n_trades"].sum(), active_weeks)
    features["fast_reentry_rate"] = _ratio((buy & window["days_since_sell"].between(1, FAST_REENTRY_DAYS)).sum(), n_sell)

    dormant = window["dormant_buy"]
    dormant_rally = _ratio((dormant & window["rally"]).sum(), dormant.sum())
    features["dormant_breakout_buy_rate"] = dormant_rally
    features["inactivity_to_rally_ratio"] = dormant_rally
    rally_opportunity = window.loc[window["rally_known"], "rally"].mean()
    features["first_buy_in_rally_bias"] = dormant_rally - rally_opportunity if np.isfinite(dormant_rally) and pd.notna(rally_opportunity) else np.nan
    features["dip_buy_lift"] = _contrast(window, "buy_day", window["dip"], window["rally_known"], MIN_CONTEXT_DAYS)
    fomo, dip = features["rally_buy_lift"], features["dip_buy_lift"]
    features["fomo_vs_dip_ratio"] = (fomo - dip) / (abs(fomo) + abs(dip)) if np.isfinite(fomo) and np.isfinite(dip) and abs(fomo) + abs(dip) > 0 else np.nan
    features["pullback_entry_bias"] = _contrast(window, "buy_day", window["pullback"], window["rally_known"], MIN_CONTEXT_DAYS)

    momentum = window["momentum"] & window["rally_known"]
    activity = window.loc[momentum, "n_buys"] + window.loc[momentum, "n_sells"]
    features["buy_sell_momentum_imbalance"] = _ratio((window.loc[momentum, "n_buys"] - window.loc[momentum, "n_sells"]).sum(), activity.sum())
    rally_sell_lift = _contrast(window, "sell_day", window["rally"], window["rally_known"], MIN_CONTEXT_DAYS)
    features["rally_sell_suppression"] = -rally_sell_lift if np.isfinite(rally_sell_lift) else np.nan

    cash = window["cash_start"].dropna()
    cash_reference = float(cash.iloc[0]) if len(cash) else np.nan
    net_buy = window["buy_cost"].sum() - window["sell_proceeds"].sum()
    features["net_buy_cash_ratio"] = _ratio(net_buy, cash_reference)
    features["portfolio_turnover_velocity"] = _ratio(window["buy_amount"].sum() + window["sell_amount"].sum(), cash_reference * max(len(window) / 7, 1))
    features["panic_sell_post_top_rate"] = _ratio((sell & window["post_top"]).sum(), window["post_top"].sum())
    return features


def _investor_windows(investor_id, raw, calendar, context, window_ends):
    daily, rows = _prepare_daily(raw, calendar, context), []
    for end in window_ends:
        start = end - pd.Timedelta(days=WINDOW_DAYS - 1)
        if end < daily.index.min():
            continue
        window = daily.loc[start:end]
        row = {
            "investor_id": investor_id, "window_start": start, "window_end": end,
            "observed_days": len(window), "n_trades": int(window["n_trades"].sum()),
            "n_buys": int(window["n_buys"].sum()), "buy_days": int(window["buy_day"].sum()),
            "sell_days": int(window["sell_day"].sum()),
            "rally_days": int((window["rally_known"] & window["rally"]).sum()),
            "calm_days": int((window["rally_known"] & ~window["rally"]).sum()),
            "buy_amount": float(window["buy_amount"].sum()),
            "sell_amount": float(window["sell_amount"].sum()),
        }
        row.update(_window_features(window))
        rows.append(row)
    return rows


def build_features(market_input, trades_input, config):
    """Build one row per investor/180-day window from in-memory raw data."""
    market = market_input.copy()
    trades = trades_input.copy()
    _check_columns(market, REQUIRED_MARKET_COLUMNS, "Market data")
    _check_columns(trades, REQUIRED_TRADE_COLUMNS, "Trade log")
    market, context = _market_context(market)
    daily_trades = _daily_trades(trades)
    calendar = pd.date_range(market.index.min(), market.index.max(), freq="D")
    if not market.index.equals(calendar):
        raise ValueError("OHLCV calendar has missing dates")

    window_days = int(config["features"].get("window_days", WINDOW_DAYS))
    step_days = int(config["features"].get("step_days", STEP_DAYS))
    if window_days != WINDOW_DAYS:
        raise ValueError("This baseline reproduces the original 180-day window")
    window_ends = pd.date_range(calendar.min() + pd.Timedelta(days=window_days - 1), calendar.max(), freq=f"{step_days}D")
    if len(window_ends) and window_ends[-1] != calendar.max():
        window_ends = window_ends.append(pd.DatetimeIndex([calendar.max()]))

    rows = []
    for investor_id, raw in daily_trades.groupby(level="investor_id", sort=True):
        rows.extend(_investor_windows(investor_id, raw.droplevel("investor_id"), calendar, context, window_ends))
    features = pd.DataFrame(rows)
    features[CANDIDATE_32] = features[CANDIDATE_32].replace([np.inf, -np.inf], np.nan)
    min_buys = int(config["features"].get("min_buys", config["features"].get("min_trades", 5)))
    min_buy_days = int(config["features"].get("min_buy_days", 3))
    features = features.loc[
        features["observed_days"].eq(window_days)
        & features["n_buys"].ge(min_buys)
        & features["buy_days"].ge(min_buy_days)
    ].sort_values(["window_end", "investor_id"]).reset_index(drop=True)
    return features


In [ ]:
def check_features(frame):
    require(not frame.empty, 'No investor-windows passed the feature filters')
    require(set(CANDIDATE_32) <= set(frame.columns), 'Missing candidate features')
    require(not np.isinf(frame[CANDIDATE_32].to_numpy()).any(), 'Infinite feature values')
    require(not frame.duplicated(['investor_id', 'window_start', 'window_end']).any(),
            'Duplicate investor-window')

windows = checked_step(
    'features', lambda: build_features(market_input, trades_input, config), check_features,
)
print(f'{len(windows):,} investor-windows; {len(CANDIDATE_32)} candidate features')
display(windows.head())


In [ ]:
FEATURE_GROUPS = {'Giá và bối cảnh thị trường': ['rally_buy_lift', 'near_high_buy_lift', 'ath_proximity_buy_ratio', 'green_streak_buy_bias', 'volume_surge_buy_lift', 'high_volatility_buy_lift', 'dip_buy_lift', 'pullback_entry_bias'], 'Vốn, quy mô và hiệu quả thoát vị thế': ['rally_size_excess', 'near_high_size_lift', 'rally_cash_excess', 'max_cash_deployment_ratio', 'all_in_frequency', 'net_buy_cash_ratio', 'portfolio_turnover_velocity', 'loss_cut_vs_profit_exit_ratio'], 'Nhịp giao dịch và tốc độ quay lại': ['rally_burst_excess', 'consecutive_buy_run_ratio', 'buy_frequency_acceleration', 'inter_buy_gap_cv', 'weekend_buy_share', 'order_intensity_per_active_week', 'fast_reentry_rate', 'rally_reactivation_excess'], 'Hướng tham gia và rút lui theo thị trường': ['dormant_breakout_buy_rate', 'inactivity_to_rally_ratio', 'first_buy_in_rally_bias', 'fomo_vs_dip_ratio', 'buy_sell_momentum_imbalance', 'rally_sell_suppression', 'sell_to_buy_day_ratio', 'panic_sell_post_top_rate']}
fig,ax=plt.subplots(figsize=(12,4))
ax.bar(range(len(FEATURE_GROUPS)),[len(v) for v in FEATURE_GROUPS.values()],color='#2878a0')
ax.set_xticks(range(4),list(FEATURE_GROUPS),rotation=15,ha='right'); ax.set(ylabel='Số ứng viên',ylim=(0,10))
show_fig('04_expanded_feature_taxonomy.png')


## 5. Phễu lọc feature

Coverage ≥ 80%, std/IQR > 0,02, loại tương quan Spearman |r| ≥ 0,70 theo thứ tự
ưu tiên cấu hình. Không cố định số feature còn lại. Sau phễu, giữ complete cases,
fit StandardScaler và PCA giữ ít nhất 90% phương sai. Các hàm dưới đây được dùng
trực tiếp cho training; không có import module nội bộ.


In [ ]:
"""Shared feature funnel and fitted scaling/PCA for pipelines and notebooks."""

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler



def select_features(frame, settings):
    """Filter by coverage/dispersion, then greedily remove Spearman redundancy.

    The configured lower-priority features are considered last. Selection uses
    pairwise available observations; complete cases are selected afterwards.
    """
    missing = set(CANDIDATE_32) - set(frame.columns)
    if missing:
        raise ValueError(f"Feature file missing model features: {sorted(missing)}")
    data = frame[CANDIDATE_32].replace([np.inf, -np.inf], np.nan)
    report = pd.DataFrame({
        "coverage": data.notna().mean(),
        "std": data.std(),
        "iqr": data.quantile(.75) - data.quantile(.25),
    })
    report["passed_coverage"] = report.coverage.ge(settings["min_coverage"])
    report["passed_dispersion"] = (
        report["std"].gt(settings["min_std"])
        & report.iqr.gt(settings["min_iqr"])
    )
    eligible = report.index[report.passed_coverage & report.passed_dispersion].tolist()
    correlation = data[eligible].corr(method="spearman")
    lower_priority = settings.get("lower_priority_features", [])
    ordered = [f for f in eligible if f not in lower_priority]
    ordered += [f for f in eligible if f in lower_priority]
    kept = []
    report["redundant_with"] = ""
    for feature in ordered:
        redundant = next((other for other in kept
                          if abs(correlation.loc[feature, other]) >= settings["max_spearman"]), None)
        if redundant is None:
            kept.append(feature)
        else:
            report.loc[feature, "redundant_with"] = redundant
    selected = [f for f in CANDIDATE_32 if f in kept]
    report["selected"] = report.index.isin(selected)
    if not selected:
        raise ValueError("Feature funnel removed all features; review preprocessing thresholds")
    report.index.name = "feature"
    return selected, report


def model_cohort(frame, feature_names):
    """Use only rows with finite values for all selected features, as in the baseline."""
    data = frame[feature_names].replace([np.inf, -np.inf], np.nan)
    cohort = frame.loc[data.notna().all(axis=1)].copy().reset_index(drop=True)
    if len(cohort) < 2:
        raise ValueError("Fewer than two complete investor-windows after feature selection")
    return cohort


def fit_preprocessing(frame, config):
    settings = config["preprocessing"]
    def check_selection(result):
        names, selection_report = result
        if not names or selection_report.loc[selection_report.selected].index.tolist() != names:
            raise ValueError("Feature funnel report differs from selected features")
        corr = frame[names].replace([np.inf, -np.inf], np.nan).corr(method="spearman").abs()
        upper = corr.to_numpy()[np.triu_indices(len(names), k=1)]
        if np.any(upper >= settings["max_spearman"]):
            raise ValueError("Feature funnel retained redundant features")

    feature_names, report = checked_step(
        "feature_funnel", lambda: select_features(frame, settings), check_selection,
    )
    cohort = checked_step(
        "complete_case", lambda: model_cohort(frame, feature_names),
        lambda result: None,  # model_cohort validates the minimum cohort size.
    )
    target = settings["pca_variance_target"]
    if not 0 < target < 1:
        raise ValueError("pca_variance_target must be between 0 and 1")
    scaler = StandardScaler()

    def check_scaled(values):
        if values.shape != (len(cohort), len(feature_names)) or not np.isfinite(values).all():
            raise ValueError("Scaling produced invalid shape or non-finite values")
        if not np.allclose(values.mean(axis=0), 0, atol=1e-7):
            raise ValueError("Scaled features are not centered")
        variance = scaler.var_
        varying = variance > 1e-12
        if not np.allclose(values[:, varying].std(axis=0), 1, atol=1e-7):
            raise ValueError("Scaled features do not have unit standard deviation")

    scaled = checked_step("scaling", lambda: scaler.fit_transform(cohort[feature_names]), check_scaled)
    pca = PCA(n_components=target, svd_solver="full", random_state=config["seed"])

    def check_pca(values):
        if values.shape[0] != len(cohort) or not 1 <= values.shape[1] <= len(feature_names):
            raise ValueError("PCA produced invalid shape")
        if not np.isfinite(values).all() or not np.isfinite(pca.explained_variance_ratio_).all():
            raise ValueError("PCA produced non-finite values or variance")
        if pca.explained_variance_ratio_.sum() + 1e-10 < target:
            raise ValueError("PCA retained variance is below configured target")

    X = checked_step("pca", lambda: pca.fit_transform(scaled), check_pca)
    preprocessor = Pipeline([("scaler", scaler), ("pca", pca)])
    return cohort, feature_names, report, preprocessor, X


In [ ]:

MODEL_FEATURES, feature_report = select_features(windows, config['preprocessing'])
display(feature_report)
funnel_counts = pd.Series({
    'Ứng viên': len(feature_report),
    'Coverage': int(feature_report.passed_coverage.sum()),
    'Std + IQR': int((feature_report.passed_coverage & feature_report.passed_dispersion).sum()),
    'Spearman': len(MODEL_FEATURES),
})
fig, ax = plt.subplots(figsize=(9, 4))
funnel_counts.plot.bar(ax=ax, color='#2878a0')
ax.set(title='Phễu lọc feature', ylabel='Số feature')
ax.tick_params(axis='x', rotation=15)
show_fig()

correlation = windows[MODEL_FEATURES].corr(method='spearman')
fig, ax = plt.subplots(figsize=(14, 11))
im = ax.imshow(correlation, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(MODEL_FEATURES)), MODEL_FEATURES, rotation=90, fontsize=7)
ax.set_yticks(range(len(MODEL_FEATURES)), MODEL_FEATURES, fontsize=7)
ax.set_title('Tương quan các feature được giữ')
fig.colorbar(im, ax=ax)
show_fig()


## 6. Scaling, PCA và training

Thử KMeans, MiniBatchKMeans và GMM trên cùng không gian PCA; chọn K theo silhouette.
Model, scaler/PCA và bảng tuning nằm trong bộ nhớ để thử nghiệm và diễn giải.


In [ ]:
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

def _score(X, labels, seed):
    rng = np.random.default_rng(seed)
    index = rng.choice(len(X), min(3000, len(X)), replace=False)
    return float(silhouette_score(X[index], np.asarray(labels)[index]))


def train_models(frame, config):
    cohort, feature_names, feature_report, preprocessor, X = fit_preprocessing(frame, config)
    settings = config["model"]
    seed = config["seed"]
    if any(k < 2 or k >= len(cohort) for k in settings["k_candidates"]):
        raise ValueError("Each candidate K must be >= 2 and smaller than the complete-case cohort")
    rows, candidates = [], {}

    for name in settings["algorithms"]:
        for k in settings["k_candidates"]:
            if name == "kmeans":
                model = KMeans(
                    n_clusters=k, n_init=settings["n_init"],
                    max_iter=settings["max_iter"], random_state=seed,
                )
            elif name == "minibatch_kmeans":
                model = MiniBatchKMeans(
                    n_clusters=k, n_init=settings["n_init"], batch_size=1024,
                    max_iter=settings["max_iter"], random_state=seed,
                )
            elif name == "gmm":
                model = GaussianMixture(
                    n_components=k, covariance_type="diag", n_init=2,
                    max_iter=settings["max_iter"], random_state=seed,
                )
            else:
                raise ValueError(f"Unsupported algorithm: {name}")

            labels = model.fit_predict(X)
            score = _score(X, labels, seed)
            rows.append({"model": name, "k": k, "silhouette": score})
            candidates[(name, k)] = model

    tuning = pd.DataFrame(rows)
    models = {}
    for name in settings['algorithms']:
        best = tuning.loc[tuning['model'].eq(name)].sort_values(
            ['silhouette', 'k'], ascending=[False, True]
        ).iloc[0]
        models[name] = candidates[(name, int(best['k']))]
    require(np.isfinite(tuning.silhouette).all(), 'Non-finite tuning scores')
    for name, model in models.items():
        labels = model.predict(X)
        require(2 <= len(np.unique(labels)) < len(cohort), f'{name}: invalid clusters')
    logger.info('step=cluster status=PASS')
    return dict(fit_data=cohort, feature_names=feature_names, feature_report=feature_report,
                preprocessor=preprocessor, X_model=X, tuning=tuning, models=models)


In [ ]:
# Model và preprocessing được giữ trong bộ nhớ để tiếp tục phân tích.
research = train_models(windows, config)
fit_data = research['fit_data']
MODEL_FEATURES = research['feature_names']
feature_report = research['feature_report']
preprocessor = research['preprocessor']
X_model = research['X_model']
X_scaled = preprocessor.named_steps['scaler'].transform(fit_data[MODEL_FEATURES])
tuning_results = research['tuning']
trained_models = research['models']
model_labels = {name: model.predict(X_model) for name, model in trained_models.items()}
pca = preprocessor.named_steps['pca']
pca_variance = pd.DataFrame({
    'component': np.arange(1, pca.n_components_ + 1),
    'explained_variance_ratio': pca.explained_variance_ratio_,
    'cumulative_variance': np.cumsum(pca.explained_variance_ratio_),
})
print(f'{len(windows):,} windows → {len(fit_data):,} complete cases; '
      f'{len(MODEL_FEATURES)} features → {X_model.shape[1]} PCs')
display(pca_variance)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(pca_variance.component, pca_variance.cumulative_variance, 'o-')
ax.axhline(config['preprocessing']['pca_variance_target'], color='#e76f51', ls='--')
ax.set(title='Phương sai tích lũy PCA', xlabel='Số PC', ylabel='Phương sai giữ lại', ylim=(0, 1.02))
show_fig()
display(tuning_results.sort_values(['model', 'k']))


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for name, scores in tuning_results.groupby('model'):
    scores = scores.sort_values('k')
    ax.plot(scores['k'], scores['silhouette'], 'o-', label=name)
ax.set(title='Chọn K theo silhouette — cùng logic với src', xlabel='K', ylabel='Silhouette')
ax.legend()
show_fig()
best_results = (tuning_results.sort_values(
    ['model', 'silhouette', 'k'], ascending=[True, False, True]
).groupby('model', as_index=False).head(1))
display(best_results)


## 7. Model evaluation

So sánh metric, quy mô và profile cụm trên cùng cohort training.
Metric và profile dùng cùng công thức với source; heatmap dùng feature chuẩn hóa trước PCA.


In [ ]:
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score


def evaluate_models(research, config):
    frame = research['fit_data']
    feature_names = research['feature_names']
    X = research['preprocessor'].transform(frame[feature_names])
    models = research['models']
    metrics, sizes, profiles = {}, [], []
    for name in config["model"]["algorithms"]:
        model = models[name]
        labels = model.predict(X)
        metrics[name] = {
            "silhouette": float(silhouette_score(X, labels, sample_size=min(3000, len(X)))),
            "calinski_harabasz": float(calinski_harabasz_score(X, labels)),
            "davies_bouldin": float(davies_bouldin_score(X, labels)),
        }
        labeled = frame.assign(model=name, cluster=labels)
        size = labeled.groupby(["model", "cluster"]).size().rename("count").reset_index()
        size["share"] = size["count"] / len(labeled)
        sizes.append(size)
        profile = labeled.groupby(["model", "cluster"])[feature_names].mean().reset_index()
        profiles.append(profile)

    for name, values in metrics.items():
        require(np.isfinite(list(values.values())).all(), f'{name}: non-finite metrics')
    sizes = pd.concat(sizes, ignore_index=True)
    profiles = pd.concat(profiles, ignore_index=True)
    for name, group in sizes.groupby('model'):
        require(group['count'].sum() == len(frame), f'{name}: cluster counts mismatch')
        require(np.isclose(group.share.sum(), 1), f'{name}: invalid cluster shares')
    logger.info('step=evaluate status=PASS')
    return metrics, sizes, profiles


In [ ]:
metrics, cluster_sizes, cluster_profiles = evaluate_models(research, config)
model_evaluation = pd.DataFrame.from_dict(metrics, orient='index')
display(model_evaluation)
display(cluster_sizes)
display(cluster_profiles)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, title in zip(axes,
    ['silhouette', 'calinski_harabasz', 'davies_bouldin'],
    ['Silhouette ↑', 'Calinski–Harabasz ↑', 'Davies–Bouldin ↓']):
    model_evaluation[metric].plot.bar(ax=ax, color='#2878a0')
    ax.set_title(title)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
show_fig()


In [ ]:
# Diễn giải tất cả mô hình bằng quy mô và đặc trưng; không vẽ hình phân tán.
# Heatmap dùng giá trị chuẩn hóa; cluster_profiles giữ đơn vị gốc.
models_for_profiles = list(model_labels.items())
for model_name, labels in models_for_profiles:
    counts = pd.Series(labels).value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(range(len(counts)), counts.values / len(labels),
           color=['#999999' if c < 0 else '#2878a0' for c in counts.index])
    ax.set_xticks(range(len(counts)), ['Chưa gán' if c < 0 else str(c) for c in counts.index], rotation=90)
    ax.set(title=f'{model_name}: quy mô các nhóm', xlabel='Mã nhóm', ylabel='Tỷ lệ cửa sổ')
    plt.tight_layout(); show_fig()
    largest = counts.idxmax()
    largest_name = 'phần chưa được gán' if largest < 0 else f'nhóm {largest}'
    profile = pd.DataFrame(X_scaled, columns=MODEL_FEATURES).assign(group=labels).groupby('group').mean()
    # Cùng thang màu giữa các nhóm của mô hình, phân trang khi có nhiều nhóm.
    color_limit = max(1., float(np.abs(profile.to_numpy()).max()))
    for start in range(0, len(profile), 12):
        part = profile.iloc[start:start+12]
        fig, ax = plt.subplots(figsize=(15, max(4, len(part)*.4)))
        im = ax.imshow(part, aspect='auto', cmap='coolwarm', vmin=-color_limit, vmax=color_limit)
        ax.set_xticks(range(len(MODEL_FEATURES)), MODEL_FEATURES, rotation=90, fontsize=8)
        ax.set_yticks(range(len(part)), ['Chưa gán' if c < 0 else str(c) for c in part.index])
        ax.set_title(f'{model_name}: đặc trưng các nhóm — phần {start//12+1}')
        fig.colorbar(im, ax=ax, label='Chênh lệch so với mức chung, tính bằng độ lệch chuẩn')
        plt.tight_layout(); show_fig()

